# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset (protein mass spectrometry data) using the `mlcroissant` library.

### Dataset Source
The dataset is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant`. We can then inspect the dataset's description, fields, and structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Print metadata summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review the dataset's available record sets, fields, and their `@id`s. With mlcroissant, you can inspect detailed schema information before extracting data.

In [ ]:
# Find available record sets
print("Record sets (@id):")
for rs in dataset.metadata.record_sets:
    print(f"  @id: {rs.id} | name: {rs.name if hasattr(rs, 'name') else 'N/A'}")

# Show fields within each record set
print("\nFields in each record set:")
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id} ({rs.name if hasattr(rs, 'name') else 'N/A'})")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, name: {f.name}, dataType: {f.dataType}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the record set `@id` and reference columns using their field `@id`s.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}:", df.columns.tolist())
    print(df.head(2))
    print()
# For the main protein abundance dataset, pick the most relevant record set (assume first for demonstration)
main_rs_id = record_set_ids[0]
print(f"Selected main record set: {main_rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric criteria, normalizing fields, and grouping by categorical attributes. All field and column references are by their `@id`.

In [ ]:
df = dataframes[main_rs_id]
# List columns and pick a numeric field for demonstration.
print("Available columns in the main record set:")
print(df.columns.tolist())
# We assume a field with '@id' like 'coverage' or 'MolecularWeight' exists (replace as appropriate)
# If you have the schema, insert the correct field `@id` here. For demonstration, we'll guess a likely candidate:
numeric_field = None
for col in df.columns:
    if 'coverage' in col.lower() or 'weight' in col.lower() or 'abundance' in col.lower() or 'peptide' in col.lower():
        numeric_field = col
        break

if numeric_field is None:
    numeric_field = df.columns[0]  # fallback to first column
print(f"Using numeric field for filtering: {numeric_field}")

try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except Exception:
    pass
threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
# Remove records with missing data in the numeric field
filtered_df = df[df[numeric_field].notnull() & (df[numeric_field] > threshold)]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records.")
print(filtered_df.head())

# Normalize numeric field
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by a categorical field, e.g., 'description', 'group', or another likely candidate
group_field = None
for col in df.columns:
    if 'desc' in col.lower() or 'sample' in col.lower() or 'condition' in col.lower():
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Plot distributions or relationships for the selected numeric field(s) using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (Filtered)")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

We successfully loaded and explored the FAIR<sup>2</sup> dataset using `mlcroissant`. We reviewed its record set structure, performed filtering and normalization on a major numeric field, and created visualizations for further interpretation. For details on all field and record set IDs, consult the Croissant schema or use the metadata inspection shown above.

Further analysis and domain-specific feature engineering can follow using these DataFrames, always referencing columns by `@id` for reproducibility.